In [44]:
import pandas as pd
import logging
from pathlib import Path

logger = logging.getLogger(__name__)

critical_columns = ["ticket_id", "date_of_purchase", "time_to_resolution"]
date_columns = ["date_of_purchase", "time_to_resolution", "first_response_time"]

def load_tickets(filepath: str) -> pd.DataFrame:
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"File not Found: {filepath}")

    if path.suffix.lower() != ".csv":
        raise ValueError(f"Expected CSV-file, received: {path.suffix}")

    logger.info(f"Downloading data from {filepath}")
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")   #нужно в строку перевести т.к тип "индекс"

    missing = [col for col in critical_columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
        
    for col in date_columns:
        if col in df.columns: #проверка есть ли значение в датафрейме   
            df[col] = pd.to_datetime(df[col], errors="coerce")  #перевод в дату и если не получается распарсить как дату, то NaT
            nulls = df[col].isna().sum()  #Считаем сколько NaT появилось после конвертации
            if nulls > 0:
                logger.warning(f"'{col}': {nulls} values are not parsed as a date")

    before = len(df)
    df = df.dropna(subset=["ticket_id"]).drop_duplicates(subset=["ticket_id"])  #df.dropna(subset=["ticket_id"]) - выбрасывает строки где пустая дата
#drop_duplicates(subset=["ticket_id"]) - выбрасывает строки где ticket_id повторяется

    dropped = before - len(df)
    if dropped:
        logger.warning(f"Deleted {dropped} rows (null ticket_id or duplicate)")

    logger.info(f"Tickets loaded: {len(df)}")
    print(df["ticket_status"].value_counts())
    return df


load_tickets ('C:/Users/katea/OneDrive/Рабочий стол/python/projects/ticket-analyzer/data/tickets.csv')





'time_to_resolution': 5700 values are not parsed as a date
'first_response_time': 2819 values are not parsed as a date


ticket_status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64


,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaT,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaT,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8464,8465,David Todd,adam28@example.net,22,Female,LG OLED,2021-12-08,Product inquiry,Installation support,My {product_purchased} is making strange noise...,Open,NaN,Low,Phone,NaT,NaT,NaN
8465,8466,Lori Davis,russell68@example.com,27,Female,Bose SoundLink Speaker,2020-02-22,Technical issue,Refund request,I'm having an issue with the {product_purchase...,Open,NaN,Critical,Email,NaT,NaT,NaN
8466,8467,Michelle Kelley,ashley83@example.org,57,Female,GoPro Action Camera,2021-08-17,Technical issue,Account access,I'm having an issue with the {product_purchase...,Closed,Eight account century nature kitchen.,High,Social media,2023-06-01 09:44:22,2023-06-01 04:31:22,3.0
8467,8468,Steven Rodriguez,fpowell@example.org,54,Male,PlayStation,2021-10-16,Product inquiry,Payment issue,I'm having an issue with the {product_purchase...,Closed,We seat culture plan.,Medium,Email,2023-06-01 18:28:24,2023-06-01 05:32:24,3.0


In [29]:
#Среднее время закрытия (MTTR) по категориям (ticket_type)
#Фильтруй только закрытые тикеты, считай разницу между time_to_resolution и date_of_purchase, группируй по ticket_type
df = load_tickets ('C:/Users/katea/OneDrive/Рабочий стол/python/projects/ticket-analyzer/data/tickets.csv')

closed = df[
    (df["ticket_status"] == "Closed") &
    (df["time_to_resolution"] >= df["first_response_time"])
].copy()

closed["mttr"] = closed["time_to_resolution"] - closed["first_response_time"]
closed["mttr_hours"] = closed["mttr"].dt.total_seconds() / 3600  #время в часах  

mttr = closed.groupby("ticket_type")["mttr_hours"].mean()

print(mttr)

'time_to_resolution': 5700 values are not parsed as a date
'first_response_time': 2819 values are not parsed as a date


ticket_status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64
ticket_type
Billing inquiry         7.010256
Cancellation request    7.693082
Product inquiry         7.676070
Refund request          8.117818
Technical issue         7.365191
Name: mttr_hours, dtype: float64


In [32]:
 #закрыт ли тикет в срок  Low — 72ч, Medium — 48ч, High — 24ч, Critical — 8ч
df = load_tickets('C:/Users/katea/OneDrive/Рабочий стол/python/projects/ticket-analyzer/data/tickets.csv')

closed = df[
    (df["ticket_status"] == "Closed") &
    (df["time_to_resolution"] >= df["first_response_time"])
].copy()

# MTTR
closed["mttr"] = closed["time_to_resolution"] - closed["first_response_time"]

# Время закрытия в часах
closed["timeclose"] = closed["mttr"].dt.total_seconds() / 3600


SLA_LIMITS = {
    "Low": 72,
    "Medium": 48,
    "High": 24,
    "Critical": 8
}

closed["sla_limit"] = closed["ticket_priority"].map(SLA_LIMITS)  #берёт колонку ticket_priority целиком и для каждого значения ищет соответствие в словаре, возвращая новую колонку sla_limit
closed["sla_status"] = closed["timeclose"] <= closed["sla_limit"] #True False

print(closed[["ticket_priority", "timeclose", "sla_limit", "sla_status"]])
print(closed[["ticket_priority", "timeclose", "SLA"]])

'time_to_resolution': 5700 values are not parsed as a date
'first_response_time': 2819 values are not parsed as a date


ticket_status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64
     ticket_priority  timeclose  sla_limit  sla_status
2                Low   6.850000         72        True
4                Low  19.683333         72        True
19               Low  19.716667         72        True
28          Critical   6.766667          8        True
29            Medium  17.483333         48        True
...              ...        ...        ...         ...
8430             Low   4.216667         72        True
8432        Critical   8.366667          8       False
8435            High   0.766667         24        True
8448          Medium   0.016667         48        True
8450             Low  17.883333         72        True

[1404 rows x 4 columns]


KeyError: "['SLA'] not in index"

In [33]:
#распределение тикетов по приоритету и по каналу (ticket_channel)

priority_distribution = df["ticket_priority"].value_counts()
channel_distribution = df["ticket_channel"].value_counts()

print(priority_distribution)
print(channel_distribution)

ticket_priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64
ticket_channel
Email           2143
Phone           2132
Social media    2121
Chat            2073
Name: count, dtype: int64


In [ ]:
#Топ-10 самых долгих тикетов
df = load_tickets('C:/Users/katea/OneDrive/Рабочий стол/python/projects/ticket-analyzer/data/tickets.csv')

closed = df[
    (df["ticket_status"] == "Closed") &
    (df["time_to_resolution"] >= df["first_response_time"])
].copy()

# MTTR
closed["mttr"] = closed["time_to_resolution"] - closed["first_response_time"]

# Время закрытия в часах
closed["timeclose"] = closed["mttr"].dt.total_seconds() / 3600


closed.sort_values("timeclose", ascending=False).head(10)

'time_to_resolution': 5700 values are not parsed as a date
'first_response_time': 2819 values are not parsed as a date


ticket_status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64


,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating,mttr,timeclose
4832,4833,Jerry Ruiz,justinlang@example.net,40,Other,Fitbit Charge,2021-08-01,Product inquiry,Payment issue,I'm having an issue with the {product_purchase...,Closed,Benefit meet of single then product doctor news.,High,Email,2023-05-31 23:41:34,2023-06-01 23:09:34,5.0,0 days 23:28:00,23.466667
5118,5119,Kimberly Wood,cynthia35@example.com,22,Other,Amazon Echo,2020-02-28,Product inquiry,Display issue,The {product_purchased} is unable to establish...,Closed,Congress else research customer.,Medium,Social media,2023-06-01 00:01:10,2023-06-01 23:08:10,2.0,0 days 23:07:00,23.116667
6649,6650,Yvette Smith,christina79@example.org,38,Female,Nest Thermostat,2020-12-01,Product inquiry,Battery life,I'm having an issue with the {product_purchase...,Closed,Law together yes total forget attention.,Critical,Social media,2023-06-01 00:51:15,2023-06-01 23:43:15,3.0,0 days 22:52:00,22.866667
6524,6525,Raymond Dickerson,jill04@example.com,41,Other,Fitbit Charge,2021-05-21,Product inquiry,Battery life,I'm having an issue with the {product_purchase...,Closed,Thing sister individual example follow person ...,Low,Social media,2023-06-01 00:18:27,2023-06-01 22:55:27,1.0,0 days 22:37:00,22.616667
2715,2716,Joseph Brown,seanclark@example.org,27,Female,PlayStation,2020-08-01,Technical issue,Product recommendation,I'm having an issue with the {product_purchase...,Closed,Himself Congress population.,Low,Email,2023-05-31 23:04:19,2023-06-01 21:41:19,3.0,0 days 22:37:00,22.616667
4665,4666,Daniel Reed,lunaandrew@example.com,27,Male,Sony 4K HDR TV,2021-12-04,Billing inquiry,Software bug,I'm having an issue with the {product_purchase...,Closed,Green hold citizen science interesting.,High,Chat,2023-06-01 01:08:52,2023-06-01 23:31:52,5.0,0 days 22:23:00,22.383333
1838,1839,Tina Brown,mary24@example.com,18,Other,HP Pavilion,2020-08-22,Refund request,Software bug,I'm having an issue with the {product_purchase...,Closed,Turn direction rule kitchen strong health.,Low,Chat,2023-05-31 23:03:45,2023-06-01 21:25:45,4.0,0 days 22:22:00,22.366667
6142,6143,Jennifer Villegas,paulastephens@example.com,35,Other,GoPro Hero,2020-10-25,Cancellation request,Hardware issue,I'm having an issue with the {product_purchase...,Closed,Form less economy evidence.,Low,Phone,2023-06-01 01:11:38,2023-06-01 23:33:38,3.0,0 days 22:22:00,22.366667
418,419,Kara Wright PhD,dawnmartinez@example.net,47,Female,Garmin Forerunner,2020-06-17,Technical issue,Software bug,I'm having an issue with the {product_purchase...,Closed,It professor television little director discus...,Low,Chat,2023-05-31 22:20:10,2023-06-01 20:41:10,2.0,0 days 22:21:00,22.350000
2648,2649,Lauren White,hclark@example.org,31,Female,Amazon Echo,2021-11-04,Billing inquiry,Hardware issue,I've forgotten my password for my {product_pur...,Closed,Care medical brother until.,Low,Social media,2023-05-31 23:25:48,2023-06-01 21:43:48,4.0,0 days 22:18:00,22.300000


In [45]:
#Тут нужна точка отсчёта по времени — логично использовать first_response_time (когда тикет реально появился в системе).
#Из datetime можно достать день недели и час через .dt.day_name() и .dt.hour.
#Дальше группировка по этим двум полям даёт количество тикетов — это и есть данные для тепловой карты.


#Группируем и считаем количество тикетов в каждой комбинации:
#.size() считает сколько строк попало в каждую группу — то есть сколько тикетов пришло  в конкретный час.
closed["hour"] = closed["first_response_time"].dt.hour
heatmap_data = closed.groupby("hour").size()
print(heatmap_data)

#.unstack() разворачивает группировку в таблицу — дни недели становятся строками, часы становятся колонками, а в ячейках — количество тикетов. 
#fill_value=0 — если для какой-то комбинации дня и часа тикетов нет, ставим 0 а не пустоту.


hour
0     101
1     116
2      87
3      91
4      86
5      89
6      82
7      71
8      78
9      66
10     59
11     65
12     47
13     41
14     56
15     48
16     32
17     25
18     33
19     27
20     12
21      8
22     23
23     61
dtype: int64


In [46]:
import pandas as pd

raw = pd.read_csv('C:/Users/katea/OneDrive/Рабочий стол/python/projects/ticket-analyzer/data/tickets.csv')
raw.columns = raw.columns.str.strip().str.lower().str.replace(" ", "_")

print(raw[raw["time_to_resolution"].isna()]["time_to_resolution"].head(10))
print(raw["ticket_status"].loc[raw["time_to_resolution"].isna()].value_counts())

0     NaN
1     NaN
5     NaN
6     NaN
7     NaN
8     NaN
9     NaN
12    NaN
13    NaN
15    NaN
Name: time_to_resolution, dtype: str
ticket_status
Pending Customer Response    2881
Open                         2819
Name: count, dtype: int64
